In [123]:
import pandas as pd
import numpy as np
import regex as re
import ast
import json

import tensorflow as tf
from sklearn.model_selection import train_test_split

In [124]:
# upload data 

X_multi = pd.read_csv('../data/processed/safedial_enriched_with_benign.csv')
X_multi['conversation'] = X_multi['conversation'].apply(json.loads)
X_train_sing = pd.read_parquet('../data/raw/wildguardmix/train/wildguard_train.parquet')
X_test_sing = pd.read_parquet('../data/raw/wildguardmix/test/wildguard_test.parquet')

In [125]:
# Harmonize the conversation format for the single turn dataset with multi turn

def build_conversation(df):
    df['conversation'] = df.apply(
        lambda row: [
            {'role': 'user', 'content': row['prompt']},
            {'role': 'assistant', 'content': row['response']}
        ],
        axis=1
    )
    return df

X_train_sing = build_conversation(X_train_sing)
X_test_sing = build_conversation(X_test_sing)

In [126]:
# Convert conversations to string format
def conversation_to_text(conversation):
    """Flatten a list of turns into a single string."""
    # Handle if still a string
    if isinstance(conversation, str):
        try:
            conversation = json.loads(conversation)
        except:
            conversation = eval(conversation)
    
    return ' '.join(
        f"{turn['role']}: {turn['content']}"
        for turn in conversation
    )
# Preprocess data
def preprocessor(conversation):
    if isinstance(conversation, list):
        text = conversation_to_text(conversation)
    else:
        text = conversation
    
    text = text.replace('\n', ' ')  # remove newlines before processing
    text = re.sub('<[^>]*>', '', text)
    emoticons = re.findall('(?::|;|=)(?:-)?(?:\)|\(|D|P)', text)
    text = (re.sub('[\W]+', ' ', text.lower()) +
            ' '.join(emoticons).replace('-', ''))
    text = text.replace('user', 'USER:') # diarize
    text = text.replace('assistant', 'ASSISTANT:')
    return text

# Encode tokens
def encode(text_tensor, label):
    text = text_tensor.numpy()[0]
    encoded_text = encoder.encode(text)
    return encoded_text, label

def encode_map_fn(text, label):
    return tf.py_function(encode, inp=[text, label],
                          Tout=(tf.int64, tf.int64))

In [127]:
# Apply functions to preprocess
X_multi['conversation'] = X_multi['conversation'].apply(
    lambda x: preprocessor(conversation_to_text(x))
)
X_train_sing['conversation'] = X_train_sing['conversation'].apply(
    lambda x: preprocessor(conversation_to_text(x))
)
X_test_sing['conversation'] = X_test_sing['conversation'].apply(
    lambda x: preprocessor(conversation_to_text(x))
)

In [128]:
# Train/ test/ validation for multi

SEED = 1234

idx = X_multi.index.to_list()
np.random.shuffle(idx)

data_shuffled = X_multi.loc[idx].reset_index(drop=True)

X = data_shuffled.drop(columns=['harm'])
y = data_shuffled[['harm']]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42)

print(len(X_train))
print(len(X_test))
print(len(X_val))

X_train.to_csv('../data/processed/multiturn_X_train.csv', index=False)
X_test.to_csv('../data/processed/multiturn_X_test.csv', index=False)
X_val.to_csv('../data/processed/multiturn_X_val.csv', index=False)


2444
815
815


In [129]:
# Train/ test/ validation for single

idx = X_train_sing.index.to_list()
np.random.shuffle(idx)

data_shuffled = X_train_sing.loc[idx].reset_index(drop=True)

X = data_shuffled

X_train_sing, X_val_sing = train_test_split(
    X_train_sing, 
    test_size=0.25, 
    random_state=SEED,
    stratify=X_train_sing['adversarial'] # maintain class balance
)

print(len(X_train_sing))
print(len(X_test_sing))
print(len(X_val_sing))

X_train_sing.to_csv('../data/processed/singleturn_X_train.csv', index=False)
X_test_sing.to_csv('../data/processed/singleturn_X_test.csv', index=False)
X_val_sing.to_csv('../data/processed/singleturn_X_val.csv', index=False)


65069
1725
21690
